# 🧠 Employee Performance Predictor
## Interactive Data Analysis Notebook

This notebook walks through the complete pipeline interactively.
Run cells one-by-one to see each step in detail.

**Pipeline:**
1. Generate synthetic HR data
2. Explore the data (EDA)
3. Preprocess & engineer features
4. Train ML models
5. Evaluate & compare models
6. Make predictions

In [ ]:
# Setup - run this first
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_style('whitegrid')
print('Libraries loaded ✅')

## Step 1: Generate Synthetic HR Dataset

In [ ]:
from src.generate_data import generate_employee_data

df = generate_employee_data(n_employees=1000)
df.to_csv('../data/employee_data.csv', index=False)
print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
# Dataset info
print('=== DATASET INFO ===')
print(df.info())
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())
print('\n=== TARGET DISTRIBUTION ===')
print(df['PerformanceLabel'].value_counts())

## Step 2: Exploratory Data Analysis (EDA)

In [ ]:
# Performance distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
counts = df['PerformanceLabel'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#C44E52','#4C72B0','#55A868'])
axes[0].set_title('Performance Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#C44E52','#4C72B0','#55A868'])
axes[1].set_title('Performance %', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
key_cols = ['TrainingHours','ProjectsCompleted','JobSatisfaction',
            'WorkLifeBalance','ManagerRating','Absences',
            'LastReviewScore','PerformanceScore']
corr = df[key_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Department breakdown
dept_perf = df.groupby(['Department','PerformanceLabel']).size().unstack(fill_value=0)
dept_perf_pct = dept_perf.div(dept_perf.sum(axis=1), axis=0) * 100
dept_perf_pct.plot(kind='bar', stacked=True, figsize=(12,6),
                   color=['#C44E52','#4C72B0','#55A868'])
plt.title('Performance by Department', fontsize=14, fontweight='bold')
plt.xlabel('Department')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=45)
plt.legend(title='Performance')
plt.tight_layout()
plt.show()

## Step 3: Preprocessing & Feature Engineering

In [ ]:
from src.preprocess import preprocess_pipeline

X_train, X_test, y_train, y_test, feature_names, scaler = preprocess_pipeline('../data/employee_data.csv')
print(f'Training set: {X_train.shape}')
print(f'Test set:     {X_test.shape}')
print(f'Features ({len(feature_names)}):')
for i, f in enumerate(feature_names):
    print(f'  {i+1:2d}. {f}')

## Step 4: Train & Compare ML Models

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import cross_val_score

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, random_state=42),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    cv  = cross_val_score(model, X_train, y_train, cv=5).mean()
    results.append({'Model': name, 'Accuracy': acc, 'F1': f1, 'CV': cv})
    print(f'{name:25s}  Acc={acc:.3f}  F1={f1:.3f}  CV={cv:.3f}')

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print(f"\n🏆 Best model: {results_df.iloc[0]['Model']}")

In [ ]:
# Model comparison chart
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df))
w = 0.25
ax.bar(x-w, results_df['Accuracy'], w, label='Accuracy', color='#4C72B0')
ax.bar(x,   results_df['F1'],       w, label='F1 Score', color='#55A868')
ax.bar(x+w, results_df['CV'],       w, label='CV Mean',  color='#DD8452')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=20, ha='right')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.axhline(0.75, color='red', linestyle='--', alpha=0.5, label='75% line')
ax.legend()
plt.tight_layout()
plt.show()

## Step 5: Best Model Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

best_name  = results_df.iloc[0]['Model']
best_model = models[best_name]
y_pred     = best_model.predict(X_test)

print(f'Best Model: {best_name}')
print(f'Accuracy:   {accuracy_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Low','Medium','High']))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low','Medium','High'],
            yticklabels=['Low','Medium','High'], ax=ax)
ax.set_title(f'Confusion Matrix - {best_name}', fontsize=13, fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

## Step 6: Make Predictions on New Employees

In [ ]:
from src.predict import load_model_and_scaler, predict_performance, simulate_new_employee

model, scaler, feature_names, metadata = load_model_and_scaler()
new_emp = simulate_new_employee()
result  = predict_performance(new_emp, model, scaler, feature_names)

print('\n' + '='*40)
print('PREDICTION RESULT')
print('='*40)
print(f"Employee:    {new_emp['EmployeeID']}")
print(f"Department:  {new_emp['Department']}")
print(f"Job Level:   {new_emp['JobLevel']}")
print(f"Predicted:   {result['predicted_performance']}")
print(f"Confidence:  {max(result['confidence'].values()):.1%}")
print('\nProbability breakdown:')
for label, prob in result['confidence'].items():
    bar = '█' * int(prob * 30)
    print(f'  {label:8s}: {bar} {prob:.1%}')